# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and analysis of a mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All dataset elements are referenced using their `@id`.

In [ ]:
# List record sets and their fields/columns by @id
record_sets = dataset.metadata.record_set

rec_set_ids = []
print('=== Record Sets ===')
for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', '<no name>')}")
    print(f"  @id: {rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', '<no @id>')}")
    rec_set_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    if rec_set_id:
        rec_set_ids.append(rec_set_id)
    fields = getattr(rs, 'field', []) if hasattr(rs, 'field') else []
    if not fields:
        fields = rs.get('field', []) if isinstance(rs, dict) else []
    print("  Fields/Columns:")
    for fld in fields:
        print(f"    - {fld['@id'] if isinstance(fld, dict) else getattr(fld, '@id', '<no @id>')} ({getattr(fld, 'name', fld.get('name', '<no name>'))})")
    print()
# Optionally preview example records for the first record set
if rec_set_ids:
    print(f"\nFirst few records from record set {rec_set_ids[0]}:")
    for i, x in enumerate(dataset.records(record_set=rec_set_ids[0])):
        if i > 4: break
        print(x)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
print("\n=== Loading all record sets into pandas DataFrames ===")
dataframes = {}

for rec_set_id in rec_set_ids:
    records = list(dataset.records(record_set=rec_set_id))
    if records:
        dataframes[rec_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {rec_set_id}.")
        print(f"Columns: {dataframes[rec_set_id].columns.tolist()}\n")
    else:
        print(f"No records found for {rec_set_id}.\n")
if dataframes:
    main_rec_set_id = rec_set_ids[0]
    print(dataframes[main_rec_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and process data for analysis using record set and field `@id`s.

In [ ]:
# Choose a numeric field and a grouping field
# You may update the field IDs below to match those listed earlier
main_rec_set_id = rec_set_ids[0] if rec_set_ids else None
df = dataframes.get(main_rec_set_id, pd.DataFrame())

# Example: Assume 'GAD-7 score' is the field with @id 'cr:GAD7_score', 'gender' has @id 'cr:gender'
numeric_field_id = 'cr:GAD7_score'  # replace with the actual @id for GAD-7
group_field_id = 'cr:gender'        # replace with the actual @id for gender

# Ensure the column exists
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Grouping by a demographic field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df[[numeric_field_id, field_norm]].head())
else:
    print(f"Field {numeric_field_id} not found in dataframe columns: {df.columns.tolist()}")

## 5. Visualization
Visualize numeric distributions and relationships.
For example, plot the distribution of GAD-7 scores, or compare groups by demographic fields.

In [ ]:
# Plot numeric field distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].dropna().hist(bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_field_id in df.columns:
        plt.figure(figsize=(7, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Numeric field {numeric_field_id} not available for visualization.")

## 6. Conclusion
Summarize key findings and relevant insights discovered during dataset exploration.

- The dataset provides rich mental health survey data tagged by unique `@id`s for fields and record sets.
- Data can be filtered and grouped, for example, by GAD-7 scores and demographic information.
- Visualizations support an understanding of score distributions and demographic relationships, useful for further analysis.

**Next steps:** Deep dive into additional survey fields, outlier and missing data handling, statistical modeling, and integration with health interventions in Kilifi County.